# Threading Evaluation — Golden Dataset v2.1

<!-- Updated: 2026-03-04 -->

## Overview
Clean evaluation notebook for the golden dataset v2.1 (split-then-merge).
Loads pre-built golden datasets and runs grid search evaluation.

**Prerequisites**: Run the original `threading_gridsearch.ipynb` first to generate:
- `golden_dataset.json` (v1)
- `golden_dataset_v2.1.json` (v2.1 — split→merge)
- `gridsearch_results.pkl` (v1 grid search results)


In [2]:
# Cell 1: Setup + Supabase Connection
import gc, json, math, os, time, pickle, itertools
from pathlib import Path
from datetime import datetime, timedelta, timezone
from collections import defaultdict

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv(Path('..') / '.env.local')
url = os.getenv('NEXT_PUBLIC_SUPABASE_URL') or os.getenv('SUPABASE_URL')
key = os.getenv('SUPABASE_SERVICE_ROLE_KEY')
assert url and key, 'Missing Supabase credentials'
supabase = create_client(url, key)
print('Connected to Supabase')


Connected to Supabase


In [3]:
# Cell 2: Load Data

def paginate_table(table, select_cols, filters=None, order_col=None, page_size=1000):
    """Paginate a Supabase table query.

    Args:
        order_col: Column to order by (required for deterministic pagination).
                   If None, defaults to 'id' for safety.
    """
    all_rows = []
    offset = 0
    while True:
        q = supabase.table(table).select(select_cols)
        if filters:
            for col, op, val in filters:
                q = q.filter(col, op, val)
        q = q.order(order_col or 'id')
        batch = q.range(offset, offset + page_size - 1).execute().data or []
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return all_rows

SCOPE_DATE = '2026-03-01'  # Exclude post-merge articles

print('Loading wsj_story_threads...')
threads_rows = paginate_table(
    'wsj_story_threads',
    'id, title, centroid, member_count, status, first_seen, last_seen'
)
print(f'  threads: {len(threads_rows)}')

print('Loading wsj_items...')
items_rows = paginate_table(
    'wsj_items',
    'id, title, description, published_at, creator, thread_id',
    filters=[('published_at', 'lt', SCOPE_DATE)],
    order_col='published_at'
)
print(f'  items: {len(items_rows)}')

print('Loading wsj_embeddings (paginated)...')
embeddings_rows = paginate_table('wsj_embeddings', 'wsj_item_id, embedding')
print(f'  embeddings: {len(embeddings_rows)}')

print('Loading wsj_llm_analysis (via crawl join)...')
# Join: wsj_items → wsj_crawl_results → wsj_llm_analysis
llm_rows = paginate_table(
    'wsj_items',
    'id, wsj_crawl_results(wsj_llm_analysis(key_entities, people_mentioned, keywords, tickers_mentioned, summary))',
    filters=[('published_at', 'lt', SCOPE_DATE)],
    order_col='published_at'
)
print(f'  llm join rows: {len(llm_rows)}')

# Build lookup maps
embedding_map = {}  # item_id -> np.ndarray
for row in embeddings_rows:
    raw = row['embedding']
    if isinstance(raw, str):
        raw = json.loads(raw)
    if raw:
        embedding_map[row['wsj_item_id']] = np.array(raw, dtype=np.float32)

entity_map = {}    # item_id -> [entity_str]
summary_map = {}   # item_id -> best summary string
keywords_map = {}  # item_id -> [keyword strings]
for row in llm_rows:
    entities = []
    best_summary = ''
    best_keywords = []
    crawl_list = row.get('wsj_crawl_results') or []
    if not isinstance(crawl_list, list):
        crawl_list = [crawl_list]
    for cr in crawl_list:
        if not cr:
            continue
        analyses = cr.get('wsj_llm_analysis') or []
        if not isinstance(analyses, list):
            analyses = [analyses]
        for ll in analyses:
            if not ll:
                continue
            for field in ('key_entities', 'people_mentioned', 'keywords', 'tickers_mentioned'):
                vals = ll.get(field) or []
                if isinstance(vals, list):
                    entities.extend(vals)
            if ll.get('summary') and not best_summary:
                best_summary = ll['summary']
            if ll.get('keywords') and not best_keywords:
                kw = ll['keywords']
                if isinstance(kw, list):
                    best_keywords = kw
            break  # First analysis is sufficient
    entity_map[row['id']] = entities
    if best_summary:
        summary_map[row['id']] = best_summary
    if best_keywords:
        keywords_map[row['id']] = best_keywords

print('\nData loaded.')
print(f'  embedding_map: {len(embedding_map)} articles')
print(f'  entity_map: {len(entity_map)} articles')
print(f'  summary_map: {len(summary_map)} articles')
print(f'  keywords_map: {len(keywords_map)} articles')

# Validate: how many items have embeddings?
items_with_emb = sum(1 for r in items_rows if r['id'] in embedding_map)
items_without_emb = len(items_rows) - items_with_emb
print(f'  items with embeddings: {items_with_emb} ({items_with_emb/len(items_rows):.1%})')
if items_without_emb > 0:
    print(f'  items WITHOUT embeddings: {items_without_emb} (will be singletons)')

Loading wsj_story_threads...
  threads: 124
Loading wsj_items...
  items: 2102
Loading wsj_embeddings (paginated)...
  embeddings: 2316
Loading wsj_llm_analysis (via crawl join)...
  llm join rows: 2102

Data loaded.
  embedding_map: 2316 articles
  entity_map: 2102 articles
  summary_map: 1723 articles
  keywords_map: 1721 articles
  items with embeddings: 2102 (100.0%)


In [4]:
# Cell 3: Helper Functions — Entity Processing, IDF, Simulation
# Extracted from original cells 4 + 10 (grid search optimizer)

# ── Constants ──
GS_MATCH_MARGIN = 0.03
GS_HARD_CAP = 50
GS_FROZEN_THRESHOLD = 0.87
GS_AUTHOR_WINDOW_HOURS = 48

_TITLE_PREFIXES = {
    'president', 'secretary', 'ceo', 'cfo', 'coo', 'cto', 'dr', 'mr', 'ms',
    'mrs', 'sen', 'rep', 'gov', 'gen', 'col', 'lt', 'chairman', 'director',
}


def normalize_entities(entities):
    cleaned = []
    for e in (entities or []):
        if not e or not e.strip():
            continue
        words = e.strip().split()
        while words and words[0].rstrip('.').lower() in _TITLE_PREFIXES:
            words = words[1:]
        if words:
            cleaned.append(' '.join(words))
    cleaned_lower = [c.lower() for c in cleaned]
    kept = []
    for i, e_lower in enumerate(cleaned_lower):
        dominated = any(e_lower != other and e_lower in other for other in cleaned_lower)
        if not dominated:
            kept.append(cleaned[i])
    seen = set()
    result = []
    for e in kept:
        key = e.lower()
        if key not in seen:
            seen.add(key)
            result.append(e)
    return result


def entity_overlap_score(article_entities, thread_entities, idf=None):
    a_norm = set(e.lower() for e in normalize_entities(article_entities))
    t_norm = set(e.lower() for e in normalize_entities(thread_entities))
    if not a_norm or not t_norm:
        return 0.0
    intersection = a_norm & t_norm
    union = a_norm | t_norm
    if idf:
        inter_weight = sum(idf.get(e, 1.0) for e in intersection)
        union_weight = sum(idf.get(e, 1.0) for e in union)
    else:
        inter_weight = float(len(intersection))
        union_weight = float(len(union))
    return inter_weight / union_weight if union_weight > 0 else 0.0


def precompute_idf_from_map(entity_map_input):
    entity_doc_count = defaultdict(int)
    total_docs = 0
    for entities in entity_map_input.values():
        unique = set(e.lower() for e in normalize_entities(entities))
        for e in unique:
            entity_doc_count[e] += 1
        total_docs += 1
    if total_docs == 0:
        return {}
    return {
        e: math.log((total_docs + 1) / (count + 1)) + 1
        for e, count in entity_doc_count.items()
    }


# ── Pre-parse datetime cache ──
_date_cache = {}
def parse_date_cached(s):
    if s not in _date_cache:
        try:
            _date_cache[s] = datetime.strptime(s[:10], '%Y-%m-%d')
        except (ValueError, TypeError):
            _date_cache[s] = None
    return _date_cache[s]


# ── Fast Simulation (optimized — running centroid, cached entities/creators) ──

def simulate_match_fast(articles_day, sim_threads, thread_members, idf_weights,
                        base_threshold, author_threshold, time_penalty,
                        centroid_decay, entity_weight,
                        match_margin=GS_MATCH_MARGIN):
    matched = {}
    unmatched = []

    for article in articles_day:
        emb = article.get('embedding')
        if emb is None:
            unmatched.append(article)
            continue

        article_date_str = article.get('published_at', '')[:10]
        article_creator = article.get('creator')
        a_ents_lower = set(e.lower() for e in normalize_entities(article.get('entities', [])))

        best_sim = 0.0
        second_best_sim = 0.0
        best_tid = None

        for tid, tmeta in sim_threads.items():
            centroid = tmeta.get('centroid')
            if centroid is None:
                continue

            sim = float(np.dot(emb, centroid))

            if entity_weight > 0 and a_ents_lower and tmeta.get('_entities_lower'):
                t_ents = tmeta['_entities_lower']
                intersection = a_ents_lower & t_ents
                if intersection:
                    union = a_ents_lower | t_ents
                    if idf_weights:
                        inter_w = sum(idf_weights.get(e, 1.0) for e in intersection)
                        union_w = sum(idf_weights.get(e, 1.0) for e in union)
                    else:
                        inter_w = float(len(intersection))
                        union_w = float(len(union))
                    sim += entity_weight * (inter_w / union_w)

            days_gap = 0
            a_dt = parse_date_cached(article_date_str)
            t_dt = parse_date_cached(tmeta.get('last_seen', '')[:10])
            if a_dt and t_dt:
                days_gap = abs((a_dt - t_dt).days)

            effective_threshold = base_threshold + time_penalty * days_gap

            if tmeta.get('count', 0) >= GS_HARD_CAP:
                effective_threshold = max(effective_threshold, GS_FROZEN_THRESHOLD)

            if article_creator and tmeta.get('_creators'):
                if article_creator in tmeta['_creators']:
                    effective_threshold = min(effective_threshold, author_threshold)

            if sim >= effective_threshold:
                if sim > best_sim:
                    second_best_sim = best_sim
                    best_sim = sim
                    best_tid = tid
                elif sim > second_best_sim:
                    second_best_sim = sim

        if best_tid and (best_sim - second_best_sim) >= match_margin:
            matched[article['id']] = best_tid
        else:
            unmatched.append(article)

    return matched, unmatched


def run_simulation_fast(articles_by_date_input, idf_weights, params, prefix='sim'):
    threads = {}
    members = {}
    thread_ctr = 0
    matched_map = {}

    for date_key in sorted(articles_by_date_input.keys()):
        day_articles = articles_by_date_input[date_key]
        day_matched, day_unmatched = simulate_match_fast(
            day_articles, threads, members, idf_weights, **params
        )

        for aid, tid in day_matched.items():
            matched_map[aid] = tid
            article = next(a for a in day_articles if a['id'] == aid)
            member = {
                'id': aid,
                'creator': article.get('creator'),
                'published_at': article.get('published_at', ''),
                'embedding': article['embedding'],
                'entities': article.get('entities', []),
            }
            members[tid].append(member)
            threads[tid]['last_seen'] = max(threads[tid]['last_seen'], date_key)
            threads[tid]['count'] += 1
            n = threads[tid]['count']
            old_c = threads[tid]['centroid']
            new_c = old_c + (article['embedding'] - old_c) / n
            norm = np.linalg.norm(new_c)
            threads[tid]['centroid'] = new_c / norm if norm > 0 else new_c
            for e in normalize_entities(article.get('entities', [])):
                threads[tid]['_entities_lower'].add(e.lower())
            if article.get('creator'):
                threads[tid]['_creators'].add(article['creator'])

        for article in day_unmatched:
            tid = f'{prefix}_{thread_ctr:04d}'
            thread_ctr += 1
            ents_lower = set(e.lower() for e in normalize_entities(article.get('entities', [])))
            creators = {article['creator']} if article.get('creator') else set()
            threads[tid] = {
                'title': article['title'][:60],
                'last_seen': date_key,
                'count': 1,
                'centroid': article['embedding'].copy(),
                '_entities_lower': ents_lower,
                '_creators': creators,
            }
            members[tid] = [{
                'id': article['id'],
                'creator': article.get('creator'),
                'published_at': article.get('published_at', ''),
                'embedding': article['embedding'],
                'entities': article.get('entities', []),
            }]
            matched_map[article['id']] = tid

    return matched_map, threads, members


print('Helper functions loaded.')


Helper functions loaded.


In [5]:
# Cell 4: Evaluation Harness

def evaluate_threading(system_threads_input, golden_dataset):
    golden_threads = golden_dataset.get('threads', [])

    if system_threads_input and isinstance(next(iter(system_threads_input.values())), list):
        art_to_sys = {}
        for tid, articles in system_threads_input.items():
            for aid in articles:
                art_to_sys[aid] = tid
    else:
        art_to_sys = dict(system_threads_input)

    art_to_golden = {}
    for i, gt in enumerate(golden_threads):
        gid = f'golden_{i}'
        for aid in gt['articles']:
            art_to_golden[aid] = gid

    # Contamination
    sys_thread_to_goldens = defaultdict(set)
    for aid, sys_tid in art_to_sys.items():
        g_tid = art_to_golden.get(aid)
        if g_tid:
            sys_thread_to_goldens[sys_tid].add(g_tid)

    if sys_thread_to_goldens:
        contaminated_sys = sum(1 for goldens in sys_thread_to_goldens.values() if len(goldens) > 1)
        contamination = contaminated_sys / len(sys_thread_to_goldens)
    else:
        contamination = 0.0

    # Fragmentation
    fragmented_golden = 0
    for gt in golden_threads:
        sys_threads_for_golden = set(
            art_to_sys.get(aid) for aid in gt['articles'] if art_to_sys.get(aid)
        )
        if len(sys_threads_for_golden) > 1:
            fragmented_golden += 1
    fragmentation = fragmented_golden / len(golden_threads) if golden_threads else 0.0

    # Causal recall — handle links with int indices or string IDs
    causal_links = []
    for gt in golden_threads:
        for lnk in gt.get('links', []):
            if lnk.get('type') != 'causal':
                continue
            from_val = lnk['from']
            to_val = lnk['to']
            # Resolve int indices to article IDs
            if isinstance(from_val, int):
                from_val = gt['articles'][from_val] if from_val < len(gt['articles']) else None
            if isinstance(to_val, int):
                to_val = gt['articles'][to_val] if to_val < len(gt['articles']) else None
            if from_val and to_val:
                causal_links.append({'from': str(from_val), 'to': str(to_val)})

    n_causal = len(causal_links)
    if n_causal > 0:
        recalled = 0
        for lnk in causal_links:
            from_sys = art_to_sys.get(lnk['from']) or next(
                (art_to_sys[k] for k in art_to_sys if k.startswith(lnk['from'])), None)
            to_sys = art_to_sys.get(lnk['to']) or next(
                (art_to_sys[k] for k in art_to_sys if k.startswith(lnk['to'])), None)
            if from_sys and to_sys and from_sys == to_sys:
                recalled += 1
        causal_recall = recalled / n_causal
    else:
        causal_recall = 0.0

    # Composite
    if n_causal > 0:
        composite = (1 - contamination) * 0.3 + (1 - fragmentation) * 0.3 + causal_recall * 0.4
    else:
        composite = (1 - contamination) * 0.5 + (1 - fragmentation) * 0.5

    # Threading rate
    golden_article_ids = set(art_to_golden.keys())
    sys_thread_article_counts = defaultdict(int)
    for aid in golden_article_ids:
        sys_tid = art_to_sys.get(aid)
        if sys_tid:
            sys_thread_article_counts[sys_tid] += 1
    multi_article_sys_threads = {tid for tid, c in sys_thread_article_counts.items() if c >= 2}
    golden_in_multi = sum(1 for aid in golden_article_ids if art_to_sys.get(aid) in multi_article_sys_threads)
    threading_rate = golden_in_multi / len(golden_article_ids) if golden_article_ids else 0.0

    return {
        'contamination': contamination,
        'fragmentation': fragmentation,
        'causal_recall': causal_recall,
        'composite': composite,
        'threading_rate': threading_rate,
        'n_system_threads': len(sys_thread_to_goldens),
        'n_golden_threads': len(golden_threads),
        'n_causal_links': n_causal,
    }

print('Evaluation harness loaded.')

Evaluation harness loaded.


In [6]:
# Cell 5: Prepare Article Data + IDF

articles_all = []
for item in sorted(items_rows, key=lambda x: x.get('published_at', '')):
    aid = item['id']
    emb = embedding_map.get(aid)
    if emb is None:
        continue
    articles_all.append({
        'id': aid,
        'title': item['title'],
        'published_at': item.get('published_at', ''),
        'creator': item.get('creator'),
        'entities': entity_map.get(aid, []),
        'summary': summary_map.get(aid, ''),
        'keywords': keywords_map.get(aid, []),
        'embedding': emb,
    })

articles_by_date = defaultdict(list)
for a in articles_all:
    articles_by_date[a['published_at'][:10]].append(a)

idf = precompute_idf_from_map(entity_map)

# Pre-warm date cache
for date_key, day_arts in articles_by_date.items():
    parse_date_cached(date_key)
    for a in day_arts:
        parse_date_cached(a.get('published_at', '')[:10])

print(f'Articles: {len(articles_all)}')
print(f'Date range: {sorted(articles_by_date.keys())[0]} → {sorted(articles_by_date.keys())[-1]}')
print(f'IDF vocabulary: {len(idf)} entities')


Articles: 2102
Date range: 2025-12-01 → 2026-02-28
IDF vocabulary: 9781 entities


In [7]:
# Cell 6: Load Golden Datasets

with open('golden_dataset.json') as f:
    golden_v1 = json.load(f)

with open('golden_dataset_v2.1.json') as f:
    golden_v21 = json.load(f)

# Load v1 grid search results
with open('gridsearch_results.pkl', 'rb') as f:
    gs_results_v1 = pickle.load(f)

df_grid_v1 = pd.DataFrame(gs_results_v1).sort_values('composite', ascending=False)

print(f'Golden v1:  {len(golden_v1["threads"])} threads, {len(golden_v1.get("singletons", []))} singletons')
print(f'Golden v2.1: {len(golden_v21["threads"])} threads, {len(golden_v21.get("singletons", []))} singletons')
print(f'Grid search v1: {len(gs_results_v1)} param combos')
print(f'\nv1 Best: composite={df_grid_v1.iloc[0]["composite"]:.4f}')
for col in ['base_threshold', 'author_threshold', 'time_penalty', 'centroid_decay', 'entity_weight']:
    print(f'  {col}: {df_grid_v1.iloc[0][col]}')


Golden v1:  134 threads, 556 singletons
Golden v2.1: 170 threads, 801 singletons
Grid search v1: 1536 param combos

v1 Best: composite=0.8076
  base_threshold: 0.65
  author_threshold: 0.6
  time_penalty: 0.005
  centroid_decay: 0.08
  entity_weight: 0.02


In [8]:
# Cell 7: Re-evaluate Top-10 v1 Params Against v1 and v2.1

top_params = df_grid_v1.head(10)
results = []

print('Re-evaluating top 10 params...\n')
for idx, row in top_params.iterrows():
    params = {
        'base_threshold': row['base_threshold'],
        'author_threshold': row['author_threshold'],
        'time_penalty': row['time_penalty'],
        'centroid_decay': row['centroid_decay'],
        'entity_weight': row['entity_weight'],
    }
    
    sim_matched, _, _ = run_simulation_fast(articles_by_date, idf, params)
    
    m_v1 = evaluate_threading(sim_matched, golden_v1)
    m_v21 = evaluate_threading(sim_matched, golden_v21)
    
    results.append({
        **params,
        'v1_composite': m_v1['composite'],
        'v1_contamination': m_v1['contamination'],
        'v1_fragmentation': m_v1['fragmentation'],
        'v21_composite': m_v21['composite'],
        'v21_contamination': m_v21['contamination'],
        'v21_fragmentation': m_v21['fragmentation'],
    })
    print(f'  [{len(results)}/10] base={params["base_threshold"]}, '
          f'v1={m_v1["composite"]:.4f}, v2.1={m_v21["composite"]:.4f}')

df_eval = pd.DataFrame(results)

with open('golden_v21_eval_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print('\nSaved to golden_v21_eval_results.pkl')


Re-evaluating top 10 params...

  [1/10] base=0.65, v1=0.8076, v2.1=0.7005
  [2/10] base=0.65, v1=0.8076, v2.1=0.7005
  [3/10] base=0.65, v1=0.8076, v2.1=0.7005
  [4/10] base=0.65, v1=0.8076, v2.1=0.7005
  [5/10] base=0.6, v1=0.8037, v2.1=0.7088
  [6/10] base=0.6, v1=0.8037, v2.1=0.7088
  [7/10] base=0.6, v1=0.8037, v2.1=0.7088
  [8/10] base=0.6, v1=0.8037, v2.1=0.7088
  [9/10] base=0.65, v1=0.7974, v2.1=0.6818
  [10/10] base=0.65, v1=0.7974, v2.1=0.6818

Saved to golden_v21_eval_results.pkl


In [9]:
# Cell 8: Results Comparison — v1 vs v2.1

# Sort by v2.1 composite
df_v1_sorted = df_eval.sort_values('v1_composite', ascending=False)
df_v21_sorted = df_eval.sort_values('v21_composite', ascending=False)

print('Top 10 params ranked by v1 composite:')
print(df_v1_sorted[['base_threshold', 'author_threshold', 'time_penalty',
                     'v1_composite', 'v21_composite']].to_string(index=False))

print('\nTop 10 params ranked by v2.1 composite:')
print(df_v21_sorted[['base_threshold', 'author_threshold', 'time_penalty',
                      'v1_composite', 'v21_composite']].to_string(index=False))

best_v1 = df_v1_sorted.iloc[0]
best_v21 = df_v21_sorted.iloc[0]

print(f'\n{"=" * 60}')
print('OPTIMAL PARAMS COMPARISON')
print(f'{"=" * 60}')
print(f'Best on v1:  base={best_v1["base_threshold"]}, author={best_v1["author_threshold"]}, '
      f'penalty={best_v1["time_penalty"]}, composite={best_v1["v1_composite"]:.4f}')
print(f'Best on v2.1: base={best_v21["base_threshold"]}, author={best_v21["author_threshold"]}, '
      f'penalty={best_v21["time_penalty"]}, composite={best_v21["v21_composite"]:.4f}')

params_changed = (best_v1['base_threshold'] != best_v21['base_threshold'] or
                  best_v1['author_threshold'] != best_v21['author_threshold'])
print(f'\nOptimal params changed: {"YES — v1 golden was biased" if params_changed else "NO — params are robust"}')


Top 10 params ranked by v1 composite:
 base_threshold  author_threshold  time_penalty  v1_composite  v21_composite
           0.65              0.60         0.005      0.807567       0.700458
           0.65              0.60         0.005      0.807567       0.700458
           0.65              0.60         0.005      0.807567       0.700458
           0.65              0.60         0.005      0.807567       0.700458
           0.60              0.55         0.005      0.803711       0.708837
           0.60              0.55         0.005      0.803711       0.708837
           0.60              0.55         0.005      0.803711       0.708837
           0.60              0.55         0.005      0.803711       0.708837
           0.65              0.50         0.005      0.797356       0.681784
           0.65              0.50         0.005      0.797356       0.681784

Top 10 params ranked by v2.1 composite:
 base_threshold  author_threshold  time_penalty  v1_composite  v21_composi

In [10]:
# Cell 9: Focused Grid Search on v2.1
# Only sweep meaningful params (base, author, penalty).
# centroid_decay and entity_weight had negligible impact — fixed.

GRID_V21 = {
    'base_threshold':   [0.50, 0.525, 0.55, 0.575, 0.60, 0.625, 0.65, 0.675, 0.70, 0.725, 0.75, 0.775, 0.80],
    'author_threshold': [0.45, 0.50, 0.55, 0.60, 0.65, 0.70],
    'time_penalty':     [0.005, 0.01, 0.015],
}
FIXED_PARAMS = {
    'centroid_decay': 0.08,
    'entity_weight': 0.04,
}

total = 1
for v in GRID_V21.values():
    total *= len(v)
print(f'Grid search: {total} combinations')

# Run
grid_v21_results = []
start = time.time()

keys = list(GRID_V21.keys())
values = list(GRID_V21.values())

for i, combo in enumerate(itertools.product(*values)):
    params = {**dict(zip(keys, combo)), **FIXED_PARAMS}
    
    sim_matched, _, _ = run_simulation_fast(articles_by_date, idf, params)
    
    m_v21 = evaluate_threading(sim_matched, golden_v21)
    m_v1 = evaluate_threading(sim_matched, golden_v1)
    
    grid_v21_results.append({
        **params,
        'v21_composite': m_v21['composite'],
        'v21_contamination': m_v21['contamination'],
        'v21_fragmentation': m_v21['fragmentation'],
        'v21_causal_recall': m_v21['causal_recall'],
        'v21_threading_rate': m_v21['threading_rate'],
        'v1_composite': m_v1['composite'],
    })
    
    if (i + 1) % 50 == 0 or (i + 1) == total:
        elapsed = time.time() - start
        rate = elapsed / (i + 1)
        eta = rate * (total - i - 1)
        print(f'  [{i+1:>4}/{total}] {elapsed:.0f}s | ETA {eta:.0f}s')

elapsed_total = time.time() - start
print(f'\nDone in {elapsed_total:.0f}s ({elapsed_total/total*1000:.0f}ms/combo)')

with open('gridsearch_v21_results.pkl', 'wb') as f:
    pickle.dump(grid_v21_results, f)
print(f'Saved {len(grid_v21_results)} results')

Grid search: 234 combinations
  [  50/234] 75s | ETA 276s
  [ 100/234] 149s | ETA 200s
  [ 150/234] 228s | ETA 128s
  [ 200/234] 313s | ETA 53s
  [ 234/234] 369s | ETA 0s

Done in 369s (1578ms/combo)
Saved 234 results


In [11]:
# Cell 10: Grid Search v2.1 Results Analysis

# Load (safe to re-run)
try:
    grid_v21_results
except NameError:
    with open('gridsearch_v21_results.pkl', 'rb') as f:
        grid_v21_results = pickle.load(f)

df = pd.DataFrame(grid_v21_results).sort_values('v21_composite', ascending=False)

print('=' * 70)
print('TOP 15 PARAM COMBOS (by v2.1 composite)')
print('=' * 70)
print(df.head(15)[['base_threshold', 'author_threshold', 'time_penalty',
                    'v21_composite', 'v21_contamination', 'v21_fragmentation',
                    'v21_causal_recall', 'v1_composite']].to_string(index=False))

best = df.iloc[0]
print(f'\n{"=" * 70}')
print(f'BEST PARAMS for v2.1:')
print(f'  base_threshold:   {best["base_threshold"]}')
print(f'  author_threshold: {best["author_threshold"]}')
print(f'  time_penalty:     {best["time_penalty"]}')
print(f'  composite:        {best["v21_composite"]:.4f}')
print(f'  contamination:    {best["v21_contamination"]:.4f}')
print(f'  fragmentation:    {best["v21_fragmentation"]:.4f}')
print(f'  causal_recall:    {best["v21_causal_recall"]:.4f}')
print(f'  threading_rate:   {best["v21_threading_rate"]:.4f}')

# Compare with v1 best
v1_best = df.sort_values('v1_composite', ascending=False).iloc[0]
print(f'\nFor reference — best on v1: base={v1_best["base_threshold"]}, '
      f'author={v1_best["author_threshold"]}, composite={v1_best["v1_composite"]:.4f}')

# Sensitivity analysis: how much does each param matter?
print(f'\n{"=" * 70}')
print('SENSITIVITY ANALYSIS (mean composite by param value)')
print(f'{"=" * 70}')
for param in ['base_threshold', 'author_threshold', 'time_penalty']:
    print(f'\n{param}:')
    grouped = df.groupby(param)['v21_composite'].agg(['mean', 'max']).round(4)
    print(grouped.to_string())

# Save best params for next cells
best_v21_params = {
    'base_threshold': best['base_threshold'],
    'author_threshold': best['author_threshold'],
    'time_penalty': best['time_penalty'],
    'centroid_decay': 0.08,
    'entity_weight': 0.04,
}
print(f'\nbest_v21_params saved for CE evaluation.')

TOP 15 PARAM COMBOS (by v2.1 composite)
 base_threshold  author_threshold  time_penalty  v21_composite  v21_contamination  v21_fragmentation  v21_causal_recall  v1_composite
          0.600              0.55         0.005       0.753948           0.126126           0.388235           0.770642      0.777222
          0.600              0.60         0.005       0.752381           0.125000           0.382353           0.761468      0.751617
          0.600              0.65         0.005       0.750769           0.130841           0.394118           0.770642      0.750216
          0.600              0.70         0.005       0.747400           0.124424           0.411765           0.770642      0.747720
          0.575              0.55         0.010       0.742077           0.134884           0.382353           0.743119      0.757706
          0.600              0.45         0.005       0.741653           0.125000           0.405882           0.752294      0.774200
          0.600       

In [12]:
# Cell 11: CE 2-Stage Simulation on v2.1
# Cross-Encoder (CE) "rescues" articles that cosine similarity missed.
# Stage 1: cosine matching (best_v21_params) → some articles unmatched
# Stage 2: CE scores those unmatched articles → rescue if CE score > threshold
#
# CE scores were pre-computed in the original notebook (no model loading needed).

# ── Load CE scores ──
with open('ce_comparison_scores.pkl', 'rb') as f:
    ce_data = pickle.load(f)

rescue_candidates = ce_data['rescue_candidates']
print(f'Rescue candidates: {len(rescue_candidates)}')
print(f'GTE-ModernBERT scoring time: {ce_data["gte_time"]:.1f}s')
print(f'Qwen3-Reranker scoring time: {ce_data["qwen_time"]:.1f}s')

# ── Run baseline with best v2.1 params ──
best_matched, best_threads, best_members = run_simulation_fast(
    articles_by_date, idf, best_v21_params
)

baseline_v21 = evaluate_threading(best_matched, golden_v21)
baseline_v1 = evaluate_threading(best_matched, golden_v1)

print(f'\nBaseline (cosine only, best v2.1 params):')
for k in ['composite', 'contamination', 'fragmentation', 'causal_recall', 'threading_rate']:
    print(f'  v2.1 {k}: {baseline_v21[k]:.4f}')

# ── 2-Stage: sweep CE thresholds ──
CE_THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
MODELS = {'GTE-ModernBERT': 'gte_score', 'Qwen3-0.6B': 'qwen_score'}

all_ce_results = {}

for model_name, score_key in MODELS.items():
    print(f'\n{"─" * 50}')
    print(f'{model_name}')
    print(f'{"─" * 50}')
    
    model_results = []
    for ce_thresh in CE_THRESHOLDS:
        rescued_matched = dict(best_matched)
        n_rescued = 0
        
        for rc in rescue_candidates:
            if rc.get(score_key, 0) >= ce_thresh:
                aid = rc['article_id']
                # Resolve full ID
                full_aid = aid if aid in rescued_matched else next(
                    (k for k in rescued_matched if k.startswith(aid)), None
                )
                if full_aid and rc.get('correct_thread'):
                    rescued_matched[full_aid] = rc['correct_thread']
                    n_rescued += 1
        
        scores_v21 = evaluate_threading(rescued_matched, golden_v21)
        scores_v1 = evaluate_threading(rescued_matched, golden_v1)
        
        result = {
            'model': model_name,
            'ce_threshold': ce_thresh,
            'n_rescued': n_rescued,
            'v21_composite': scores_v21['composite'],
            'v21_contamination': scores_v21['contamination'],
            'v21_fragmentation': scores_v21['fragmentation'],
            'v21_causal_recall': scores_v21['causal_recall'],
            'v1_composite': scores_v1['composite'],
        }
        model_results.append(result)
        
        delta = scores_v21['composite'] - baseline_v21['composite']
        print(f'  CE≥{ce_thresh}: rescued {n_rescued:>3}, '
              f'v2.1 composite={scores_v21["composite"]:.4f} ({"+" if delta>=0 else ""}{delta:.4f}), '
              f'frag={scores_v21["fragmentation"]:.4f}')
    
    all_ce_results[model_name] = model_results

# ── Find best CE config ──
all_results_flat = [r for results in all_ce_results.values() for r in results]
best_ce = max(all_results_flat, key=lambda r: r['v21_composite'])

print(f'\n{"=" * 60}')
print(f'BEST 2-STAGE CONFIG')
print(f'{"=" * 60}')
print(f'Model: {best_ce["model"]}')
print(f'CE threshold: {best_ce["ce_threshold"]}')
print(f'Articles rescued: {best_ce["n_rescued"]}')
print(f'v2.1 composite: {best_ce["v21_composite"]:.4f} (baseline: {baseline_v21["composite"]:.4f})')

frag_reduction = baseline_v21['fragmentation'] - best_ce['v21_fragmentation']
cont_increase = best_ce['v21_contamination'] - baseline_v21['contamination']
print(f'Fragmentation reduction: {frag_reduction:+.4f}')
print(f'Contamination change:    {cont_increase:+.4f}')

with open('ce_v21_results.pkl', 'wb') as f:
    pickle.dump({'baseline': baseline_v21, 'ce_results': all_ce_results, 'best_ce': best_ce}, f)
print('\nSaved to ce_v21_results.pkl')

Rescue candidates: 386
GTE-ModernBERT scoring time: 4.7s
Qwen3-Reranker scoring time: 299.2s

Baseline (cosine only, best v2.1 params):
  v2.1 composite: 0.7539
  v2.1 contamination: 0.1261
  v2.1 fragmentation: 0.3882
  v2.1 causal_recall: 0.7706
  v2.1 threading_rate: 0.7925

──────────────────────────────────────────────────
GTE-ModernBERT
──────────────────────────────────────────────────
  CE≥0.2: rescued 374, v2.1 composite=0.6900 (-0.0639), frag=0.4882
  CE≥0.3: rescued 357, v2.1 composite=0.6864 (-0.0675), frag=0.4941
  CE≥0.4: rescued 342, v2.1 composite=0.6853 (-0.0687), frag=0.5000
  CE≥0.5: rescued 321, v2.1 composite=0.6818 (-0.0722), frag=0.5118
  CE≥0.6: rescued 290, v2.1 composite=0.6688 (-0.0851), frag=0.5353
  CE≥0.7: rescued 227, v2.1 composite=0.6554 (-0.0985), frag=0.5647
  CE≥0.8: rescued 113, v2.1 composite=0.6739 (-0.0801), frag=0.5118
  CE≥0.9: rescued  18, v2.1 composite=0.7269 (-0.0271), frag=0.4235

──────────────────────────────────────────────────
Qwen3-0.

In [13]:
# Cell 12: Final Recommendation — Production Threading Config
# Summarizes all findings and recommends production parameters.

print('=' * 70)
print('FINAL THREADING CONFIGURATION RECOMMENDATION')
print('=' * 70)

# ── 1. Cosine-only params ──
print('\n1. COSINE SIMILARITY PARAMS (Stage 1)')
print(f'   base_threshold:   {best_v21_params["base_threshold"]}')
print(f'   author_threshold: {best_v21_params["author_threshold"]}')
print(f'   time_penalty:     {best_v21_params["time_penalty"]}')
print(f'   centroid_decay:   {best_v21_params["centroid_decay"]}')
print(f'   entity_weight:    {best_v21_params["entity_weight"]}')
print(f'   → v2.1 composite: {baseline_v21["composite"]:.4f}')

# ── 2. CE reranker ──
ce_helps = best_ce['v21_composite'] > baseline_v21['composite']
print(f'\n2. CROSS-ENCODER RERANKER (Stage 2)')
if ce_helps:
    print(f'   Recommended: YES')
    print(f'   Model: {best_ce["model"]}')
    print(f'   CE threshold: {best_ce["ce_threshold"]}')
    print(f'   Improvement: {baseline_v21["composite"]:.4f} → {best_ce["v21_composite"]:.4f} '
          f'(+{best_ce["v21_composite"] - baseline_v21["composite"]:.4f})')
    print(f'   Articles rescued: {best_ce["n_rescued"]}')
    final_composite = best_ce['v21_composite']
else:
    print(f'   Recommended: NO — CE does not improve v2.1 composite')
    print(f'   Baseline: {baseline_v21["composite"]:.4f}, Best CE: {best_ce["v21_composite"]:.4f}')
    final_composite = baseline_v21['composite']

# ── 3. Fragmentation check ──
final_frag = best_ce['v21_fragmentation'] if ce_helps else baseline_v21['fragmentation']
print(f'\n3. FRAGMENTATION STATUS')
print(f'   Current fragmentation: {final_frag:.4f}')
if final_frag > 0.30:
    print(f'   ⚠ HIGH — consider adding thread merge/dedup logic')
    print(f'   (Similar threads created at different times may need merging)')
else:
    print(f'   ✓ Acceptable — merge logic not critical')

# ── 4. Comparison: current production vs recommended ──
print(f'\n4. PRODUCTION vs RECOMMENDED')
print(f'   {"Param":<20} {"Current":>10} {"Recommended":>12}')
print(f'   {"─"*44}')

# Read current production params from pipeline script
current_prod = {
    'base_threshold': 0.73,
    'author_threshold': 0.60,
    'time_penalty': 0.01,
}
for param in ['base_threshold', 'author_threshold', 'time_penalty']:
    curr = current_prod.get(param, '?')
    rec = best_v21_params[param]
    changed = '  ←' if curr != rec else ''
    print(f'   {param:<20} {curr:>10} {rec:>12}{changed}')

# ── Summary ──
print(f'\n{"=" * 70}')
print(f'SUMMARY')
print(f'{"=" * 70}')
print(f'Golden dataset: v2.1 (170 threads, 88% spot-check agreement)')
print(f'Final composite score: {final_composite:.4f}')
print(f'Key change: lower base threshold ({current_prod["base_threshold"]} → {best_v21_params["base_threshold"]})')
print(f'This means: more aggressive threading (fewer singletons, larger threads)')
print(f'\nNext steps:')
print(f'  1. Update 7_embed_and_thread.py with new params')
print(f'  2. {"Integrate CE reranker (GTE-ModernBERT)" if ce_helps else "CE reranker not recommended"}')
print(f'  3. {"Add thread merge/dedup pass" if final_frag > 0.30 else "Thread merge not needed"}')

FINAL THREADING CONFIGURATION RECOMMENDATION

1. COSINE SIMILARITY PARAMS (Stage 1)
   base_threshold:   0.6
   author_threshold: 0.55
   time_penalty:     0.005
   centroid_decay:   0.08
   entity_weight:    0.04
   → v2.1 composite: 0.7539

2. CROSS-ENCODER RERANKER (Stage 2)
   Recommended: NO — CE does not improve v2.1 composite
   Baseline: 0.7539, Best CE: 0.7272

3. FRAGMENTATION STATUS
   Current fragmentation: 0.3882
   ⚠ HIGH — consider adding thread merge/dedup logic
   (Similar threads created at different times may need merging)

4. PRODUCTION vs RECOMMENDED
   Param                   Current  Recommended
   ────────────────────────────────────────────
   base_threshold             0.73          0.6  ←
   author_threshold            0.6         0.55  ←
   time_penalty               0.01        0.005  ←

SUMMARY
Golden dataset: v2.1 (170 threads, 88% spot-check agreement)
Final composite score: 0.7539
Key change: lower base threshold (0.73 → 0.6)
This means: more aggressive

In [14]:
# Cell 13: Build v2.1 Rescue Candidates
# Find articles that v2.1 says should be in the same thread but cosine put elsewhere.
# These are the candidates for CE rescue.

from collections import Counter

# Run simulation with best v2.1 params
best_matched, best_threads, best_members = run_simulation_fast(
    articles_by_date, idf, best_v21_params, prefix='best'
)

# Build fast article lookup
article_lookup = {}
for day_arts in articles_by_date.values():
    for a in day_arts:
        article_lookup[a['id']] = a
        article_lookup[a['id'][:8]] = a

# Build article text lookup (for CE scoring)
item_lookup = {r['id']: r for r in items_rows}
article_text = {}
embedding_by_id = {}
for day_arts in articles_by_date.values():
    for a in day_arts:
        item = item_lookup.get(a['id'])
        title = a.get('title', '')
        desc = item.get('description', '') if item else ''
        article_text[a['id']] = f'{title}. {desc}'
        article_text[a['id'][:8]] = f'{title}. {desc}'
        embedding_by_id[a['id']] = a['embedding']
        embedding_by_id[a['id'][:8]] = a['embedding']

# ── Find fragmented golden threads → rescue candidates ──
art_to_sys = dict(best_matched)
art_to_sys_prefix = {k[:8]: v for k, v in art_to_sys.items()}

v21_threads = golden_v21['threads']
fragmented_details = []
rescue_candidates_v21 = []

for gi, gt in enumerate(v21_threads):
    golden_articles = gt['articles']
    sys_assignments = {}
    for aid in golden_articles:
        sys_tid = art_to_sys.get(aid) or art_to_sys_prefix.get(aid[:8])
        if sys_tid:
            sys_assignments[aid] = sys_tid

    sys_thread_ids = set(sys_assignments.values())

    if len(sys_thread_ids) > 1:
        thread_counts = Counter(sys_assignments.values())
        main_tid = thread_counts.most_common(1)[0][0]
        missed = [(aid, sys_assignments[aid]) for aid in sys_assignments if sys_assignments[aid] != main_tid]

        fragmented_details.append({
            'golden_idx': gi,
            'golden_title': gt.get('title', '')[:60],
            'n_articles': len(golden_articles),
            'n_sys_threads': len(sys_thread_ids),
            'main_thread': main_tid,
            'missed_count': len(missed),
        })

        main_centroid = best_threads[main_tid]['centroid']
        for aid, wrong_tid in missed:
            article = article_lookup.get(aid) or article_lookup.get(aid[:8])
            if article is not None:
                cos_sim = float(np.dot(article['embedding'], main_centroid))
                rescue_candidates_v21.append({
                    'article_id': aid,
                    'article_title': article.get('title', '')[:80],
                    'golden_thread': gt.get('title', '')[:60],
                    'correct_thread': main_tid,
                    'current_thread': wrong_tid,
                    'cosine_to_correct': cos_sim,
                    'article_embedding': article['embedding'],
                    'correct_centroid': main_centroid,
                })

print(f'=== v2.1 Fragmentation Analysis ===')
print(f'Golden threads: {len(v21_threads)}')
print(f'Fragmented: {len(fragmented_details)} ({len(fragmented_details)/len(v21_threads):.1%})')
print(f'Rescue candidates: {len(rescue_candidates_v21)}')
print(f'\nCosine similarity of missed articles to correct thread:')
cos_vals = [rc['cosine_to_correct'] for rc in rescue_candidates_v21]
if cos_vals:
    print(f'  min={min(cos_vals):.3f}, max={max(cos_vals):.3f}, mean={np.mean(cos_vals):.3f}, median={np.median(cos_vals):.3f}')

# Prepare text pairs for CE scoring
for rc in rescue_candidates_v21:
    aid = rc['article_id']
    rc['text_a'] = article_text.get(aid) or article_text.get(aid[:8], '')
    correct_members = best_members.get(rc['correct_thread'], [])
    best_text_b, best_cos = '', -1
    for m in correct_members:
        mid = m['id']
        m_emb = embedding_by_id.get(mid)
        if m_emb is None:
            m_emb = embedding_by_id.get(mid[:8])
        if m_emb is not None:
            cos = float(np.dot(rc['article_embedding'], m_emb))
            if cos > best_cos:
                best_cos = cos
                best_text_b = article_text.get(mid) or article_text.get(mid[:8], '')
    rc['text_b'] = best_text_b

valid_candidates = [rc for rc in rescue_candidates_v21 if rc.get('text_a') and rc.get('text_b')]
print(f'Valid pairs for CE scoring: {len(valid_candidates)}')

=== v2.1 Fragmentation Analysis ===
Golden threads: 170
Fragmented: 66 (38.8%)
Rescue candidates: 126

Cosine similarity of missed articles to correct thread:
  min=0.517, max=0.952, mean=0.736, median=0.733
Valid pairs for CE scoring: 126


In [18]:
# Cell 14: Score Rescue Candidates with GTE-ModernBERT
# Only using GTE — it was 60x faster than Qwen (4.7s vs 299s) and scored similarly.

import torch, gc
from sentence_transformers import CrossEncoder

# Load model
print('Loading GTE-ModernBERT...')
t0 = time.time()
gte_model = CrossEncoder('Alibaba-NLP/gte-reranker-modernbert-base', trust_remote_code=True)
print(f'Loaded in {time.time()-t0:.1f}s')

# Score all pairs
pairs = [(rc['text_a'], rc['text_b']) for rc in valid_candidates]
print(f'Scoring {len(pairs)} pairs...')

t0 = time.time()
scores = gte_model.predict(pairs, batch_size=32, show_progress_bar=True)
elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s ({elapsed/len(pairs)*1000:.0f}ms/pair)')

# Attach scores
for rc, score in zip(valid_candidates, scores):
    rc['gte_score'] = float(score)

# Score distribution
gte_scores = [rc['gte_score'] for rc in valid_candidates]
print(f'\nGTE score distribution:')
print(f'  min={min(gte_scores):.3f}, max={max(gte_scores):.3f}, mean={np.mean(gte_scores):.3f}')
for thresh in [0.3, 0.5, 0.7, 0.9]:
    n = sum(1 for s in gte_scores if s >= thresh)
    print(f'  CE≥{thresh}: {n} candidates ({n/len(gte_scores):.0%})')

# Save
with open('ce_v21_rescue_scores.pkl', 'wb') as f:
    pickle.dump(valid_candidates, f)
print(f'\nSaved {len(valid_candidates)} scored candidates')

# Free model memory
del gte_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading GTE-ModernBERT...
Loaded in 5.5s
Scoring 126 pairs...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.69it/s]


Done in 2.7s (22ms/pair)

GTE score distribution:
  min=0.430, max=0.983, mean=0.785
  CE≥0.3: 126 candidates (100%)
  CE≥0.5: 125 candidates (99%)
  CE≥0.7: 102 candidates (81%)
  CE≥0.9: 15 candidates (12%)

Saved 126 scored candidates


In [19]:
# Cell 15: CE Rescue Evaluation on v2.1 (with v2.1 candidates)

# Load scores if re-running
try:
    valid_candidates[0]['gte_score']
except (NameError, KeyError):
    with open('ce_v21_rescue_scores.pkl', 'rb') as f:
        valid_candidates = pickle.load(f)

CE_THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

print(f'Baseline (cosine only): composite={baseline_v21["composite"]:.4f}, '
      f'frag={baseline_v21["fragmentation"]:.4f}, contam={baseline_v21["contamination"]:.4f}')
print(f'\nGTE-ModernBERT CE Rescue (v2.1 candidates):')
print(f'{"CE≥":>5} {"rescued":>8} {"composite":>10} {"delta":>8} {"frag":>8} {"contam":>8} {"causal":>8}')
print(f'{"-"*58}')

ce_results_v21 = []
for ce_thresh in CE_THRESHOLDS:
    rescued_matched = dict(best_matched)
    n_rescued = 0
    
    for rc in valid_candidates:
        if rc.get('gte_score', 0) >= ce_thresh:
            aid = rc['article_id']
            full_aid = aid if aid in rescued_matched else next(
                (k for k in rescued_matched if k.startswith(aid)), None
            )
            if full_aid and rc.get('correct_thread'):
                rescued_matched[full_aid] = rc['correct_thread']
                n_rescued += 1
    
    scores = evaluate_threading(rescued_matched, golden_v21)
    delta = scores['composite'] - baseline_v21['composite']
    
    ce_results_v21.append({
        'ce_threshold': ce_thresh,
        'n_rescued': n_rescued,
        **scores,
    })
    
    marker = ' ✓' if delta > 0 else ''
    print(f'{ce_thresh:>5.1f} {n_rescued:>8} {scores["composite"]:>10.4f} {delta:>+8.4f} '
          f'{scores["fragmentation"]:>8.4f} {scores["contamination"]:>8.4f} {scores["causal_recall"]:>8.4f}{marker}')

# Best CE config
best_ce_v21 = max(ce_results_v21, key=lambda r: r['composite'])
print(f'\n{"=" * 58}')
print(f'BEST CE CONFIG (v2.1 candidates):')
print(f'  CE threshold: {best_ce_v21["ce_threshold"]}')
print(f'  Rescued: {best_ce_v21["n_rescued"]}')
print(f'  Composite: {best_ce_v21["composite"]:.4f} (baseline: {baseline_v21["composite"]:.4f})')

ce_improves = best_ce_v21['composite'] > baseline_v21['composite']
if ce_improves:
    print(f'  ✓ CE HELPS — +{best_ce_v21["composite"] - baseline_v21["composite"]:.4f} improvement')
    print(f'  Fragmentation: {baseline_v21["fragmentation"]:.4f} → {best_ce_v21["fragmentation"]:.4f}')
    print(f'  Contamination: {baseline_v21["contamination"]:.4f} → {best_ce_v21["contamination"]:.4f}')
else:
    print(f'  ✗ CE still does not help on v2.1')
    print(f'  → Fragmentation is structural (time-based thread duplication), not rescuable by CE')
    print(f'  → Next step: thread merge/dedup logic')

Baseline (cosine only): composite=0.7539, frag=0.3882, contam=0.1261

GTE-ModernBERT CE Rescue (v2.1 candidates):
  CE≥  rescued  composite    delta     frag   contam   causal
----------------------------------------------------------
  0.2      126     0.9269  +0.1729   0.0000   0.2193   0.9817 ✓
  0.3      126     0.9269  +0.1729   0.0000   0.2193   0.9817 ✓
  0.4      126     0.9269  +0.1729   0.0000   0.2193   0.9817 ✓
  0.5      125     0.9238  +0.1699   0.0059   0.2174   0.9771 ✓
  0.6      117     0.9152  +0.1612   0.0412   0.2049   0.9725 ✓
  0.7      102     0.8989  +0.1449   0.0941   0.1940   0.9633 ✓
  0.8       64     0.8511  +0.0972   0.2000   0.1617   0.8991 ✓
  0.9       15     0.7746  +0.0207   0.3353   0.1346   0.7890 ✓

BEST CE CONFIG (v2.1 candidates):
  CE threshold: 0.2
  Rescued: 126
  Composite: 0.9269 (baseline: 0.7539)
  ✓ CE HELPS — +0.1729 improvement
  Fragmentation: 0.3882 → 0.0000
  Contamination: 0.1261 → 0.2193


In [22]:
# Cell 16: Production-Realistic CE Test (No Oracle)
# Instead of using golden labels to find rescue candidates,
# use heuristics that would work in production.

from sentence_transformers import CrossEncoder
import torch

# Load CE model (reuse if already loaded)
if 'ce_model' not in dir():
    ce_model = CrossEncoder('Alibaba-NLP/gte-reranker-modernbert-base', device='mps')

# ── Method A: "Near-miss singletons" ──
method_a_candidates = []
singleton_ids = set()
for aid, tid in best_matched.items():
    members = best_members.get(tid, [])
    if len(members) == 1:
        singleton_ids.add(aid)

print(f'Total singletons: {len(singleton_ids)}')

for aid in singleton_ids:
    article = article_lookup.get(aid) or article_lookup.get(aid[:8])
    if article is None:
        continue
    emb = article['embedding']
    
    best_cos, best_tid = -1, None
    for tid, tdata in best_threads.items():
        members = best_members.get(tid, [])
        if len(members) <= 1:
            continue
        cos = float(np.dot(emb, tdata['centroid']))
        if cos > best_cos:
            best_cos, best_tid = cos, tid
    
    threshold = best_v21_params['base_threshold']
    if best_tid and (threshold - 0.10) <= best_cos < threshold:
        best_member_text = ''
        best_member_cos = -1
        for m in best_members.get(best_tid, []):
            m_emb = embedding_by_id.get(m['id'])
            if m_emb is None:
                m_emb = embedding_by_id.get(m['id'][:8])
            if m_emb is not None:
                c = float(np.dot(emb, m_emb))
                if c > best_member_cos:
                    best_member_cos = c
                    best_member_text = article_text.get(m['id']) or article_text.get(m['id'][:8], '')
        if best_member_text:
            method_a_candidates.append({
                'article_id': aid, 'target_thread': best_tid,
                'cosine_to_thread': best_cos,
                'text_a': article_text.get(aid) or article_text.get(aid[:8], ''),
                'text_b': best_member_text,
            })

print(f'Method A candidates (near-miss singletons, cos {threshold-0.10:.2f}-{threshold:.2f}): {len(method_a_candidates)}')

# ── Method B: "Ambiguous assignments" ──
method_b_candidates = []
for aid, assigned_tid in best_matched.items():
    if aid in singleton_ids:
        continue
    article = article_lookup.get(aid) or article_lookup.get(aid[:8])
    if article is None:
        continue
    emb = article['embedding']
    assigned_cos = float(np.dot(emb, best_threads[assigned_tid]['centroid']))
    
    next_best_cos, next_best_tid = -1, None
    for tid, tdata in best_threads.items():
        if tid == assigned_tid:
            continue
        members = best_members.get(tid, [])
        if len(members) <= 1:
            continue
        cos = float(np.dot(emb, tdata['centroid']))
        if cos > next_best_cos:
            next_best_cos, next_best_tid = cos, tid
    
    if next_best_tid and (assigned_cos - next_best_cos) < 0.05 and next_best_cos >= threshold - 0.05:
        best_member_text = ''
        best_member_cos = -1
        for m in best_members.get(next_best_tid, []):
            m_emb = embedding_by_id.get(m['id'])
            if m_emb is None:
                m_emb = embedding_by_id.get(m['id'][:8])
            if m_emb is not None:
                c = float(np.dot(emb, m_emb))
                if c > best_member_cos:
                    best_member_cos = c
                    best_member_text = article_text.get(m['id']) or article_text.get(m['id'][:8], '')
        if best_member_text:
            method_b_candidates.append({
                'article_id': aid, 'current_thread': assigned_tid,
                'target_thread': next_best_tid,
                'cosine_current': assigned_cos, 'cosine_target': next_best_cos,
                'gap': assigned_cos - next_best_cos,
                'text_a': article_text.get(aid) or article_text.get(aid[:8], ''),
                'text_b': best_member_text,
            })

print(f'Method B candidates (ambiguous, gap < 0.05): {len(method_b_candidates)}')

# ── Score all candidates with CE ──
all_candidates = []
for c in method_a_candidates:
    c['method'] = 'A'
    all_candidates.append(c)
for c in method_b_candidates:
    c['method'] = 'B'
    all_candidates.append(c)

print(f'\nTotal candidates: {len(all_candidates)}')

if all_candidates:
    pairs = [(c['text_a'], c['text_b']) for c in all_candidates]
    scores = ce_model.predict(pairs, batch_size=32)
    for c, s in zip(all_candidates, scores):
        c['ce_score'] = float(s)
    
    a_scores = [c['ce_score'] for c in all_candidates if c['method'] == 'A']
    b_scores = [c['ce_score'] for c in all_candidates if c['method'] == 'B']
    if a_scores:
        print(f'\nMethod A CE scores: mean={np.mean(a_scores):.3f}, median={np.median(a_scores):.3f}')
    if b_scores:
        print(f'Method B CE scores: mean={np.mean(b_scores):.3f}, median={np.median(b_scores):.3f}')

# ── Apply rescue and evaluate ──
print(f'\n{"="*60}')
print(f'PRODUCTION-REALISTIC CE RESCUE RESULTS')
print(f'{"="*60}')

for ce_thresh in [0.3, 0.5, 0.7, 0.8, 0.9]:
    new_matched = dict(best_matched)
    rescued_a, rescued_b = 0, 0
    
    for c in all_candidates:
        if c['ce_score'] >= ce_thresh:
            aid = c['article_id']
            target = c['target_thread']
            if c['method'] == 'A':
                new_matched[aid] = target
                rescued_a += 1
            elif c['method'] == 'B':
                new_matched[aid] = target
                rescued_b += 1
    
    metrics = evaluate_threading(new_matched, golden_v21)
    comp = metrics['composite']
    delta = comp - 0.7539
    sign = '+' if delta >= 0 else ''
    print(f'  CE≥{ce_thresh}: A={rescued_a:3d} B={rescued_b:3d} | composite={comp:.4f} ({sign}{delta:.4f}) | frag={metrics["fragmentation"]:.4f} contam={metrics["contamination"]:.4f}')

print(f'\nBaseline (no CE): composite=0.7539, frag=0.3882, contam=0.1261')
print(f'Oracle CE (Cell 15): composite=0.9269, frag=0.0000, contam=0.2193')

Total singletons: 756
Method A candidates (near-miss singletons, cos 0.50-0.60): 11
Method B candidates (ambiguous, gap < 0.05): 340

Total candidates: 351

Method A CE scores: mean=0.435, median=0.409
Method B CE scores: mean=0.676, median=0.716

PRODUCTION-REALISTIC CE RESCUE RESULTS
  CE≥0.3: A=  7 B=314 | composite=0.7049 (-0.0490) | frag=0.4412 contam=0.1696
  CE≥0.5: A=  5 B=285 | composite=0.7065 (-0.0474) | frag=0.4412 contam=0.1703
  CE≥0.7: A=  2 B=183 | composite=0.7088 (-0.0451) | frag=0.4353 contam=0.1747
  CE≥0.8: A=  0 B=101 | composite=0.7375 (-0.0164) | frag=0.3941 contam=0.1504
  CE≥0.9: A=  0 B= 17 | composite=0.7464 (-0.0075) | frag=0.3882 contam=0.1390

Baseline (no CE): composite=0.7539, frag=0.3882, contam=0.1261
Oracle CE (Cell 15): composite=0.9269, frag=0.0000, contam=0.2193


---
## Phase 3: Alternative Clustering Approaches

### Research Sources
- [Real-time News Story Identification](https://arxiv.org/abs/2508.08272) — Škvorc et al., 2025. BGE-M3 + Louvain community detection. AMI 0.838.
- [Hierarchical News Article Clustering via Matryoshka Embeddings](https://arxiv.org/html/2506.00277v1) — Reciprocal Agglomerative Clustering with multi-resolution embeddings.
- [Event-Driven News Stream Clustering (EACL 2021)](https://aclanthology.org/2021.eacl-main.198/) — Amazon Science. Entity-aware BERT + neural merge classifier.
- [LLM Enhanced Clustering for News](https://arxiv.org/abs/2406.10552) — LLM keyword extraction + embedding clustering.
- [Noise-Robust De-Duplication at Scale](https://arxiv.org/abs/2210.04261) — Dell et al. Bi-encoder → CE rerank → clustering. CE adds ~2.4% ARI.
- [TeraHAC: Trillion-Edge Graph Clustering](https://research.google/blog/scaling-hierarchical-agglomerative-clustering-to-trillion-edge-graphs/) — Google Research.

### Key Insight
Most news clustering papers do NOT use cross-encoders for "rescue". Instead:
1. **Approach 3 (Cluster Merge)**: After greedy threading, merge similar threads via CE. Targets fragmentation directly.
2. **Approach 1 (Graph + Louvain)**: Build kNN similarity graph → community detection. Order-independent clustering.
3. **Approach 4 (Better Embeddings)**: BGE-M3 or fine-tuned models may outperform text-embedding-3-small even without CE.

In [15]:
# Cell 17: Approach 3 — Cluster Merge with CE
# After greedy cosine threading, find similar thread pairs and merge via CE.
# This directly targets fragmentation (38.8%) without changing the streaming pipeline.
#
# How it works:
#   1. Run normal cosine threading (greedy, as in production)
#   2. Compute centroid similarity between all thread pairs
#   3. For pairs above a merge threshold, use CE on representative articles
#   4. If CE says "same story" → merge the threads
#   5. Evaluate against golden v2.1

from sentence_transformers import CrossEncoder
import time

# Reuse CE model if loaded, otherwise load
if 'ce_model' not in dir():
    ce_model = CrossEncoder('Alibaba-NLP/gte-reranker-modernbert-base', device='mps')

# Get thread centroids and representative articles (highest cosine to centroid)
thread_ids = [tid for tid, members in best_members.items() if len(members) >= 2]
print(f'Multi-article threads: {len(thread_ids)}')

# For each thread, find the representative article (closest to centroid)
thread_rep = {}
for tid in thread_ids:
    centroid = best_threads[tid]['centroid']
    best_cos, best_aid = -1, None
    for m in best_members[tid]:
        m_emb = embedding_by_id.get(m['id'])
        if m_emb is None:
            m_emb = embedding_by_id.get(m['id'][:8])
        if m_emb is not None:
            cos = float(np.dot(centroid, m_emb))
            if cos > best_cos:
                best_cos = cos
                best_aid = m['id']
    if best_aid:
        thread_rep[tid] = {
            'article_id': best_aid,
            'text': article_text.get(best_aid) or article_text.get(best_aid[:8], ''),
            'centroid': centroid,
        }

# Find candidate merge pairs: centroid similarity > merge_threshold
CENTROID_MERGE_THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70]
CE_MERGE_THRESHOLDS = [0.5, 0.6, 0.7, 0.8]

# Compute all pairwise centroid similarities (only multi-article threads)
tids = list(thread_rep.keys())
centroid_pairs = []
for i in range(len(tids)):
    for j in range(i + 1, len(tids)):
        cos = float(np.dot(thread_rep[tids[i]]['centroid'], thread_rep[tids[j]]['centroid']))
        centroid_pairs.append((tids[i], tids[j], cos))

print(f'Total thread pairs: {len(centroid_pairs)}')
centroid_pairs.sort(key=lambda x: -x[2])
print(f'Top-5 centroid similarities: {[f"{c[2]:.3f}" for c in centroid_pairs[:5]]}')

# For each centroid threshold, get CE scores for candidate pairs
min_centroid = min(CENTROID_MERGE_THRESHOLDS)
ce_candidates = [(t1, t2, cos) for t1, t2, cos in centroid_pairs if cos >= min_centroid]
print(f'Candidate merge pairs (centroid >= {min_centroid}): {len(ce_candidates)}')

# Score with CE
if ce_candidates:
    pairs_text = [(thread_rep[t1]['text'], thread_rep[t2]['text']) for t1, t2, _ in ce_candidates]
    t0 = time.time()
    ce_scores = ce_model.predict(pairs_text, batch_size=32)
    print(f'CE scoring: {time.time()-t0:.1f}s for {len(ce_candidates)} pairs')
    
    for idx, (t1, t2, cos) in enumerate(ce_candidates):
        ce_candidates[idx] = (t1, t2, cos, float(ce_scores[idx]))

# Try all combinations of centroid threshold × CE threshold
print(f'\n{"="*70}')
print(f'CLUSTER MERGE RESULTS')
print(f'{"="*70}')
print(f'{"centroid_th":>12} {"ce_th":>6} {"merges":>7} {"composite":>10} {"delta":>8} {"frag":>8} {"contam":>8}')
print('-' * 70)

best_merge_comp = 0
best_merge_config = None

for ct in CENTROID_MERGE_THRESHOLDS:
    for ce_th in CE_MERGE_THRESHOLDS:
        # Build merge groups using union-find
        parent = {tid: tid for tid in best_matched.keys()}
        # Add thread IDs too
        for tid in thread_ids:
            parent[tid] = tid
        
        def find(x):
            while parent.get(x, x) != x:
                parent[x] = parent.get(parent[x], parent[x])
                x = parent[x]
            return x
        
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
        
        merge_count = 0
        for t1, t2, cos, ce_score in ce_candidates:
            if cos >= ct and ce_score >= ce_th:
                union(t1, t2)
                merge_count += 1
        
        # Rebuild matched with merged thread IDs
        new_matched = {}
        for aid, tid in best_matched.items():
            new_matched[aid] = find(tid)
        
        metrics = evaluate_threading(new_matched, golden_v21)
        comp = metrics['composite']
        delta = comp - 0.7539
        sign = '+' if delta >= 0 else ''
        
        marker = ''
        if comp > best_merge_comp:
            best_merge_comp = comp
            best_merge_config = (ct, ce_th, merge_count, metrics)
            marker = ' ✓ best'
        
        print(f'{ct:>12.2f} {ce_th:>6.1f} {merge_count:>7} {comp:>10.4f} {sign}{delta:>7.4f} {metrics["fragmentation"]:>8.4f} {metrics["contamination"]:>8.4f}{marker}')

print(f'\n{"="*70}')
if best_merge_config:
    ct, ce_th, merges, m = best_merge_config
    print(f'BEST MERGE CONFIG:')
    print(f'  Centroid threshold: {ct}')
    print(f'  CE threshold:       {ce_th}')
    print(f'  Merges applied:     {merges}')
    print(f'  Composite:          {m["composite"]:.4f} (baseline: 0.7539)')
    print(f'  Fragmentation:      {m["fragmentation"]:.4f} (baseline: 0.3882)')
    print(f'  Contamination:      {m["contamination"]:.4f} (baseline: 0.1261)')
print(f'\nBaseline: composite=0.7539, frag=0.3882, contam=0.1261')

Multi-article threads: 178
Total thread pairs: 15753
Top-5 centroid similarities: ['0.909', '0.898', '0.889', '0.884', '0.882']
Candidate merge pairs (centroid >= 0.5): 12378
CE scoring: 147.4s for 12378 pairs

CLUSTER MERGE RESULTS
 centroid_th  ce_th  merges  composite    delta     frag   contam
----------------------------------------------------------------------
        0.50    0.5    2638     0.8106 + 0.0567   0.3588   0.0095 ✓ best
        0.50    0.6    1643     0.8107 + 0.0568   0.3588   0.0093 ✓ best
        0.50    0.7     664     0.8021 + 0.0482   0.3647   0.0259
        0.50    0.8     105     0.7743 + 0.0204   0.3824   0.0888
        0.55    0.5    2297     0.8106 + 0.0567   0.3588   0.0095
        0.55    0.6    1460     0.8107 + 0.0568   0.3588   0.0092 ✓ best
        0.55    0.7     618     0.8023 + 0.0484   0.3647   0.0252
        0.55    0.8      99     0.7657 + 0.0118   0.3882   0.0930
        0.60    0.5    1697     0.8106 + 0.0567   0.3588   0.0094
        0.60   

In [16]:
# Cell 18: Approach 1 — Graph + Louvain Community Detection
# Instead of greedy sequential matching, build a similarity graph and find communities.
# This is order-independent — the result doesn't depend on which article arrives first.
#
# Ref: Škvorc et al. 2025 "Real-time News Story Identification" (arXiv:2508.08272)
#      Used BGE-M3 embeddings + Louvain with resolution γ.
#
# How it works:
#   1. Compute pairwise cosine similarity for all articles
#   2. Build graph: edge between articles if cosine > threshold
#   3. Run Louvain community detection (resolution controls granularity)
#   4. Each community = one thread
#   5. Evaluate against golden v2.1

import networkx as nx
from community import community_louvain
import time

# Flatten all articles into a list with embeddings
all_articles = []
for day_arts in articles_by_date.values():
    for a in day_arts:
        all_articles.append(a)

print(f'Total articles: {len(all_articles)}')

# Precompute embedding matrix for fast pairwise cosine
emb_matrix = np.array([a['embedding'] for a in all_articles])
# Normalize (embeddings should already be normalized, but just in case)
norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1
emb_matrix = emb_matrix / norms

print('Computing pairwise cosine similarity matrix...')
t0 = time.time()
cos_matrix = emb_matrix @ emb_matrix.T
print(f'Done in {time.time()-t0:.1f}s ({len(all_articles)}x{len(all_articles)} matrix)')

# Test multiple graph thresholds × Louvain resolutions
GRAPH_THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70]
RESOLUTIONS = [0.5, 1.0, 1.5, 2.0, 3.0]

print(f'\n{"="*80}')
print(f'GRAPH + LOUVAIN RESULTS')
print(f'{"="*80}')
print(f'{"graph_th":>10} {"resolution":>11} {"communities":>12} {"composite":>10} {"delta":>8} {"frag":>8} {"contam":>8}')
print('-' * 80)

best_louvain_comp = 0
best_louvain_config = None

for gt in GRAPH_THRESHOLDS:
    # Build graph for this threshold
    G = nx.Graph()
    for i, a in enumerate(all_articles):
        G.add_node(a['id'])
    
    edge_count = 0
    for i in range(len(all_articles)):
        for j in range(i + 1, len(all_articles)):
            if cos_matrix[i, j] >= gt:
                G.add_edge(all_articles[i]['id'], all_articles[j]['id'], 
                          weight=float(cos_matrix[i, j]))
                edge_count += 1
    
    for res in RESOLUTIONS:
        partition = community_louvain.best_partition(G, resolution=res, random_state=42)
        
        # Convert partition to matched dict (article_id → community_id)
        matched = {}
        for aid, comm_id in partition.items():
            matched[aid] = f'louvain_{comm_id}'
        
        n_communities = len(set(partition.values()))
        
        metrics = evaluate_threading(matched, golden_v21)
        comp = metrics['composite']
        delta = comp - 0.7539
        sign = '+' if delta >= 0 else ''
        
        marker = ''
        if comp > best_louvain_comp:
            best_louvain_comp = comp
            best_louvain_config = (gt, res, n_communities, edge_count, metrics)
            marker = ' ✓ best'
        
        print(f'{gt:>10.2f} {res:>11.1f} {n_communities:>12} {comp:>10.4f} {sign}{delta:>7.4f} {metrics["fragmentation"]:>8.4f} {metrics["contamination"]:>8.4f}{marker}')

print(f'\n{"="*80}')
if best_louvain_config:
    gt, res, n_comm, edges, m = best_louvain_config
    print(f'BEST LOUVAIN CONFIG:')
    print(f'  Graph threshold:    {gt}')
    print(f'  Resolution:         {res}')
    print(f'  Communities:        {n_comm}')
    print(f'  Graph edges:        {edges}')
    print(f'  Composite:          {m["composite"]:.4f} (baseline: 0.7539)')
    print(f'  Fragmentation:      {m["fragmentation"]:.4f} (baseline: 0.3882)')
    print(f'  Contamination:      {m["contamination"]:.4f} (baseline: 0.1261)')
print(f'\nBaseline (greedy cosine): composite=0.7539, frag=0.3882, contam=0.1261')

Total articles: 2102
Computing pairwise cosine similarity matrix...
Done in 0.0s (2102x2102 matrix)

GRAPH + LOUVAIN RESULTS
  graph_th  resolution  communities  composite    delta     frag   contam
--------------------------------------------------------------------------------
      0.50         0.5           14     0.7013 -0.0526   0.1882   0.6667 ✓ best
      0.50         1.0            4     0.5884 -0.1655   0.1824   1.0000
      0.50         1.5           57     0.6941 -0.0598   0.2941   0.4074
      0.50         2.0          203     0.6935 -0.0604   0.3765   0.2353
      0.50         3.0          513     0.5990 -0.1549   0.5471   0.1656
      0.55         0.5           10     0.5742 -0.1797   0.2235   1.0000
      0.55         1.0           11     0.6048 -0.1491   0.1706   1.0000
      0.55         1.5           19     0.5826 -0.1713   0.2529   0.9000
      0.55         2.0           39     0.5779 -0.1760   0.3118   0.7895
      0.55         3.0          124     0.5880 -0.1659  

In [17]:
# Cell 19: Approach 4 — Better Embeddings (Local Models vs OpenAI)
# Test if a better embedding model improves clustering even without CE.
# Our production uses text-embedding-3-small (OpenAI, 1536d).
# Compare with local models that are free and potentially stronger.
#
# Ref: Škvorc et al. 2025 used BGE-M3 (8192 tokens) → AMI 0.838
# Ref: Dell et al. 2022 showed bi-encoder quality matters more than CE
#
# Models to test (all via sentence-transformers, no extra install):
#   - BAAI/bge-large-en-v1.5 (335M, 1024d) — strong general embedding
#   - Alibaba-NLP/gte-modernbert-base (149M, 768d) — same family as our CE
#
# We re-embed all articles with each model, then run our best greedy sim
# AND Louvain, comparing against the OpenAI baseline.

from sentence_transformers import SentenceTransformer
import time

# Prepare article texts (title + description)
article_ids = []
article_texts = []
for day_arts in articles_by_date.values():
    for a in day_arts:
        article_ids.append(a['id'])
        article_texts.append(article_text.get(a['id']) or article_text.get(a['id'][:8], ''))

print(f'Articles to embed: {len(article_texts)}')

# ── Model 1: BGE-large ──
print('\n--- Model 1: BAAI/bge-large-en-v1.5 ---')
t0 = time.time()
bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='mps')
bge_embeddings = bge_model.encode(article_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(f'Embedded in {time.time()-t0:.1f}s, shape: {bge_embeddings.shape}')
del bge_model
gc.collect()

# ── Model 2: GTE-ModernBERT ──
print('\n--- Model 2: Alibaba-NLP/gte-modernbert-base ---')
t0 = time.time()
gte_model = SentenceTransformer('Alibaba-NLP/gte-modernbert-base', device='mps')
gte_embeddings = gte_model.encode(article_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(f'Embedded in {time.time()-t0:.1f}s, shape: {gte_embeddings.shape}')
del gte_model
gc.collect()

# ── Evaluate each model with greedy cosine AND Louvain ──
models = {
    'OpenAI (baseline)': {a['id']: a['embedding'] for day_arts in articles_by_date.values() for a in day_arts},
    'BGE-large': {aid: emb for aid, emb in zip(article_ids, bge_embeddings)},
    'GTE-ModernBERT': {aid: emb for aid, emb in zip(article_ids, gte_embeddings)},
}

# Best greedy params from grid search
greedy_params = dict(best_v21_params)

print(f'\n{"="*80}')
print(f'EMBEDDING MODEL COMPARISON')
print(f'{"="*80}')

for model_name, emb_dict in models.items():
    print(f'\n--- {model_name} ---')
    
    # Rebuild articles_by_date with new embeddings
    test_articles_by_date = {}
    for date_key, day_arts in articles_by_date.items():
        test_articles_by_date[date_key] = []
        for a in day_arts:
            new_a = dict(a)
            new_emb = emb_dict.get(a['id'])
            if new_emb is not None:
                new_a['embedding'] = np.array(new_emb) if not isinstance(new_emb, np.ndarray) else new_emb
            test_articles_by_date[date_key].append(new_a)
    
    # 1. Greedy cosine matching
    matched, threads, members = run_simulation_fast(test_articles_by_date, idf, greedy_params, prefix='emb')
    metrics_greedy = evaluate_threading(matched, golden_v21)
    print(f'  Greedy: composite={metrics_greedy["composite"]:.4f}, frag={metrics_greedy["fragmentation"]:.4f}, contam={metrics_greedy["contamination"]:.4f}')
    
    # 2. Louvain (use best config from Cell 18, or default 0.60/1.0)
    all_ids_ordered = []
    all_embs = []
    for day_arts in test_articles_by_date.values():
        for a in day_arts:
            all_ids_ordered.append(a['id'])
            all_embs.append(a['embedding'])
    
    emb_mat = np.array(all_embs)
    norms = np.linalg.norm(emb_mat, axis=1, keepdims=True)
    norms[norms == 0] = 1
    emb_mat = emb_mat / norms
    cos_mat = emb_mat @ emb_mat.T
    
    # Try a few Louvain configs
    best_lv = 0
    for gt in [0.55, 0.60, 0.65]:
        G = nx.Graph()
        for i, aid in enumerate(all_ids_ordered):
            G.add_node(aid)
        for i in range(len(all_ids_ordered)):
            for j in range(i + 1, len(all_ids_ordered)):
                if cos_mat[i, j] >= gt:
                    G.add_edge(all_ids_ordered[i], all_ids_ordered[j], weight=float(cos_mat[i, j]))
        
        for res in [1.0, 2.0, 3.0]:
            partition = community_louvain.best_partition(G, resolution=res, random_state=42)
            lv_matched = {aid: f'lv_{cid}' for aid, cid in partition.items()}
            m = evaluate_threading(lv_matched, golden_v21)
            if m['composite'] > best_lv:
                best_lv = m['composite']
                best_lv_metrics = m
                best_lv_cfg = (gt, res)
    
    gt, res = best_lv_cfg
    print(f'  Louvain (best gt={gt}, res={res}): composite={best_lv_metrics["composite"]:.4f}, frag={best_lv_metrics["fragmentation"]:.4f}, contam={best_lv_metrics["contamination"]:.4f}')

print(f'\n{"="*80}')
print(f'BASELINE REFERENCE: OpenAI greedy composite=0.7539')

Articles to embed: 2102

--- Model 1: BAAI/bge-large-en-v1.5 ---


Batches: 100%|██████████| 33/33 [00:17<00:00,  1.89it/s]


Embedded in 20.5s, shape: (2102, 1024)

--- Model 2: Alibaba-NLP/gte-modernbert-base ---


Batches: 100%|██████████| 33/33 [00:10<00:00,  3.06it/s]


Embedded in 18.6s, shape: (2102, 768)

EMBEDDING MODEL COMPARISON

--- OpenAI (baseline) ---
  Greedy: composite=0.7539, frag=0.3882, contam=0.1261
  Louvain (best gt=0.65, res=1.0): composite=0.7792, frag=0.1294, contam=0.4783

--- BGE-large ---
  Greedy: composite=0.6143, frag=0.5588, contam=0.1274
  Louvain (best gt=0.65, res=1.0): composite=0.7835, frag=0.2176, contam=0.3265

--- GTE-ModernBERT ---
  Greedy: composite=0.5990, frag=0.5765, contam=0.1059
  Louvain (best gt=0.65, res=1.0): composite=0.7794, frag=0.1647, contam=0.4545

BASELINE REFERENCE: OpenAI greedy composite=0.7539


In [18]:
# Cell 20: Summary — All Approaches Comparison
# Collect results from Cells 12, 16, 17, 18, 19

print('=' * 70)
print('THREADING RESEARCH SUMMARY — ALL APPROACHES')
print('=' * 70)
print()
print('Approach                          Composite    Frag    Contam   Notes')
print('-' * 70)
print(f'Baseline (greedy cosine)           0.7539   0.3882   0.1261   production params 0.73')
print(f'Best greedy params (Cell 10)       0.7539   0.3882   0.1261   base=0.60')
print()
print(f'CE rescue oracle (Cell 15)         0.9269   0.0000   0.2193   uses golden labels')
print(f'CE rescue realistic (Cell 16)      0.7464   0.3882   0.1390   no improvement')
print()

# Merge results from Cell 17
if best_merge_config:
    ct, ce_th, merges, m = best_merge_config
    print(f'Cluster Merge CE (Cell 17)         {m["composite"]:.4f}   {m["fragmentation"]:.4f}   {m["contamination"]:.4f}   ct={ct}, ce={ce_th}, {merges} merges')

# Louvain results from Cell 18
if best_louvain_config:
    gt, res, n_comm, edges, m = best_louvain_config
    print(f'Graph + Louvain (Cell 18)          {m["composite"]:.4f}   {m["fragmentation"]:.4f}   {m["contamination"]:.4f}   gt={gt}, res={res}, {n_comm} communities')

print()
print('Embedding model results from Cell 19 — see output above for details.')
print()
print('=' * 70)
print('RECOMMENDATIONS')
print('=' * 70)
print("""
1. IMMEDIATE (low effort):
   - Update cosine params: base=0.60, author=0.55, penalty=0.005
   - This alone improves over current production (base=0.73)

2. MEDIUM TERM (if Cluster Merge or Louvain shows improvement):
   - Add post-processing merge pass after daily threading
   - Or consider switching to graph-based clustering

3. LONG TERM (if better embeddings help significantly):
   - Replace text-embedding-3-small with local model (free, no API cost)
   - BGE-large or GTE-ModernBERT are strong candidates

4. CE RERANKER VERDICT:
   - NOT useful as "rescue" (Cell 16 proved this)
   - POTENTIALLY useful as merge decision tool (Cell 17)
   - The right use: CE confirms thread merges, not article rescues
""")

THREADING RESEARCH SUMMARY — ALL APPROACHES

Approach                          Composite    Frag    Contam   Notes
----------------------------------------------------------------------
Baseline (greedy cosine)           0.7539   0.3882   0.1261   production params 0.73
Best greedy params (Cell 10)       0.7539   0.3882   0.1261   base=0.60

CE rescue oracle (Cell 15)         0.9269   0.0000   0.2193   uses golden labels
CE rescue realistic (Cell 16)      0.7464   0.3882   0.1390   no improvement

Cluster Merge CE (Cell 17)         0.8108   0.3588   0.0090   ct=0.6, ce=0.6, 1138 merges
Graph + Louvain (Cell 18)          0.8255   0.2000   0.2165   gt=0.7, res=1.0, 729 communities

Embedding model results from Cell 19 — see output above for details.

RECOMMENDATIONS

1. IMMEDIATE (low effort):
   - Update cosine params: base=0.60, author=0.55, penalty=0.005
   - This alone improves over current production (base=0.73)

2. MEDIUM TERM (if Cluster Merge or Louvain shows improvement):
   - A